In [25]:
import os
import json
import pandas as pd
from typing import Dict, Any

# =============================
# 0) 파일 경로 설정
# =============================
SINGLE_FILE = "llm_outputs_100/프리미엄 동원참치액 900g_100_personas.jsonl"  # 제품+페르소나 합본 JSONL
# SINGLE_FILE = "llm_outputs_100/동원맛참 고소참기름 135g_100_personas.jsonl"  # 제품+페르소나 합본 JSONL
COMPETITOR_FILE_CLEANED = "competitor_prices_cleaned_final.csv"
OUTPUT_FILE = "llm_prompt/프리미엄 동원참치액 900g_prompts_for_llm.jsonl"

# =============================
# 1) 카테고리 매핑/헬퍼
# =============================
FINAL_CATEGORY_MAPPING = {
    '참치 > 참치캔 > 라이트스탠다드참치': '참치캔',
    '참치 > 참치캔 > 가미참치': '참치캔',
    '조미소스 > 조미료 > 액상조미료': '참치액',
    '우유류 > 발효유 > 호상-중대용량': '그릭요거트',
    '축산 > 햄/소시지 > 캔햄': '캔햄',
    '수산 > 수산캔 > 번데기/골뱅이/꽁치': '캔햄',
    '우유류 > 커피 > 커피-CUP': 'RTD_액상커피'
}

def get_main_category(detailed_category: str) -> str | None:
    if not detailed_category:
        return None
    if detailed_category in FINAL_CATEGORY_MAPPING:
        return FINAL_CATEGORY_MAPPING[detailed_category]
    if '참치캔' in detailed_category: return '참치캔'
    if '참치액' in detailed_category or '액상조미료' in detailed_category: return '참치액'
    if '그릭' in detailed_category or '발효유' in detailed_category: return '그릭요거트'
    if '캔햄' in detailed_category: return '캔햄'
    if '커피' in detailed_category: return 'RTD_액상커피'
    return None

ATTRIBUTE_TRANSLATION = {
    "health_orientation_scaled": "건강 지향성",
    "price_sensitivity_scaled": "가격 민감도",
    "premium_orientation_scaled": "프리미엄 지향성",
    "variety_seeking_scaled": "다양성 추구",
    "cooking_convenience_scaled": "요리 편리성",
    "brand_loyalty_scaled": "브랜드 충성도",
    "hmr_preference_scaled": "HMR 선호도",
}

def get_most_important_attribute(persona_info: Dict[str, Any]) -> str:
    """
    페르소나의 구매 성향 속성 중 중요도(가중치/값)가 가장 높은 속성명을 한국어로 반환.
    - v가 dict이면: weight 우선, 없으면 value 사용
    - v가 숫자(float/int)면: 그 숫자를 중요도 점수로 간주
    - NaN/None은 0 처리
    """
    attrs = persona_info.get('attributes', {}) or {}
    best_key = None
    best_score = float('-inf')

    # dict 또는 list 등 다양한 케이스를 방어적으로 처리
    if isinstance(attrs, dict):
        items = attrs.items()
    else:
        # 예외 케이스: 만약 리스트 형태라면 name/value 구조를 예상
        try:
            items = [(a.get('name'), a.get('value')) for a in attrs if isinstance(a, dict)]
        except Exception:
            items = []

    for k, v in items:
        if not k or '_scaled' not in k:
            continue

        # 중요도 점수 계산
        score = 0.0
        if isinstance(v, dict):
            w = v.get('weight')
            val = v.get('value')
            if isinstance(w, (int, float)):
                score = float(w)
            elif isinstance(val, (int, float)):
                score = float(val)
        elif isinstance(v, (int, float)):
            score = float(v)

        # NaN 방지
        try:
            if pd.isna(score):
                score = 0.0
        except Exception:
            pass

        if score > best_score:
            best_score = score
            best_key = k

    if not best_key:
        return "(분석 불가)"
    return ATTRIBUTE_TRANSLATION.get(best_key, best_key)

# =============================
# 2) 합본 JSONL에서 제품/페르소나 추출
# =============================
def derive_product_name_from_filename(filename: str) -> str:
    """파일명 '제품명_100_personas.jsonl' 형태에서 제품명 부분만 추출"""
    base = os.path.basename(filename)
    name = os.path.splitext(base)[0]
    # 뒤쪽의 '_###_personas' 같은 꼬리표 제거
    tokens = name.split('_')
    if len(tokens) >= 2 and tokens[-1].lower().endswith('personas'):
        return '_'.join(tokens[:-1])
    if len(tokens) >= 2 and tokens[-1].isdigit():
        return '_'.join(tokens[:-1])
    return name

PRODUCT_KEYS = ['brand', 'product_name', 'category', 'features', 'targeted_consumer', 'price_text', 'advertise_info']

def extract_product_info(record: Dict[str, Any], filename_fallback: str) -> Dict[str, Any]:
    """
    레코드에서 제품 정보를 유연하게 추출.
    - record['product_info'] 또는 record['product']에 있으면 사용
    - 아니면 record의 최상위 키들에서 가능한 값 수집
    - 부족하면 파일명에서 제품명 추론
    """
    # 1) 명시적 product 객체
    if isinstance(record.get('product_info'), dict):
        prod = record['product_info']
    elif isinstance(record.get('product'), dict):
        prod = record['product']
    else:
        # 2) 최상위 키에서 줍줍
        prod = {k: record.get(k) for k in PRODUCT_KEYS if k in record}

    # 3) 누락값 보정
    if not prod.get('product_name'):
        prod['product_name'] = derive_product_name_from_filename(filename_fallback)

    # 카테고리 보정(없으면 제품명으로 추론)
    if not prod.get('category'):
        pn = str(prod.get('product_name', ''))
        if '요거트' in pn or '그릭' in pn:
            prod['category'] = '우유류 > 발효유 > 호상-중대용량'  # 매핑을 통해 '그릭요거트'가 됨

    return prod

def extract_persona_info(record: Dict[str, Any]) -> Dict[str, Any]:
    """
    레코드에서 페르소나 정보 추출.
    - record['persona'] or record['persona_info']가 있으면 사용
    - 없으면 record 자체가 페르소나라 가정
    """
    if isinstance(record.get('persona'), dict):
        return record['persona']
    if isinstance(record.get('persona_info'), dict):
        return record['persona_info']
    return record

# =============================
# 3) 시장 경쟁사 맥락 로드
# =============================
def load_market_context(path_csv: str) -> Dict[str, Dict[str, str]]:
    try:
        df = pd.read_csv(path_csv)
        ctx = {}
        for category, group in df.groupby('category'):
            min_price = group['price_per_100g'].min()
            max_price = group['price_per_100g'].max()
            ctx[category] = {
                "competitor_price_range_per_100g": f"{int(min_price):,}원 ~ {int(max_price):,}원"
            }
        print(f"경쟁사 시장 데이터 불러오기 완료. (카테고리: {list(ctx.keys())})")
        return ctx
    except FileNotFoundError:
        print(f"오류: '{path_csv}' 파일을 찾을 수 없습니다. 시장 맥락 없이 진행합니다.")
        return {}

# =============================
# 4) 프롬프트 생성
# =============================
INCLUDE_REASON = True  # True로 바꾸면 reason 포함
def create_advanced_prompt(product_info: Dict[str, Any],
                           persona_info: Dict[str, Any],
                           market_context: Dict[str, Dict[str, str]]) -> tuple[str, str]:
    system_prompt = (
        "당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. "
        "오직 최종 JSON 객체 하나만 출력하세요. 코드블록/설명/추가 텍스트 금지."
    )

    PRODUCT_KEYS = ['brand', 'product_name', 'category', 'features', 'targeted_consumer', 'price_text', 'advertise_info']
    product_subset = {k: product_info.get(k) for k in PRODUCT_KEYS}
    product_str = json.dumps(product_subset, ensure_ascii=False, indent=4)
    persona_str = json.dumps(persona_info, ensure_ascii=False, indent=4)

    main_category_key = get_main_category(product_info.get('category', '') or '')
    context_data = market_context.get(main_category_key, {}) if main_category_key else {}
    context_str = json.dumps(context_data, ensure_ascii=False, indent=4) if context_data else "{}"

    most_important_attr = get_most_important_attribute(persona_info)
    persona_key_str = str(persona_info.get('persona_key'))

    # OUTPUT FORMAT (reason 토글)
    if INCLUDE_REASON:
        output_format = f"""{{
  "product_name": "{product_info.get('product_name','')}",
  "persona_key": "{persona_key_str}",
  "purchase_behavior_prediction": {{
    "purchase_probability_pct": 0,
    "monthly_avg_purchase_qty": 0,
    "reason": "<200자 이하 한 줄 요약>"
  }}
}}"""
    else:
        output_format = f"""{{
  "product_name": "{product_info.get('product_name','')}",
  "persona_key": "{persona_key_str}",
  "purchase_behavior_prediction": {{
    "purchase_probability_pct": 0,
    "monthly_avg_purchase_qty": 0
  }}
}}"""

    # 사용자 프롬프트(내부 사고만, 출력은 JSON만)
    user_prompt = f"""# INSTRUCTION (Compact)
아래 데이터를 바탕으로 **내부적으로만** 분석하세요(사고과정 출력 금지).
최종 출력은 위 # OUTPUT FORMAT과 **정확히 동일한 키 구조의 JSON 한 덩어리**만 반환합니다.
- 'purchase_probability_pct'는 0~100 **숫자**
- 'monthly_avg_purchase_qty'는 0 이상 **숫자**
- {'reason은 200자 이하 한 줄, 줄바꿈 금지.' if INCLUDE_REASON else 'reason은 절대 포함하지 마세요.'}

# INPUT DATA
## 1) 제품
{product_str}

## 2) 페르소나
{persona_str}

## 3) 시장 경쟁 환경
{context_str}

# 분석 기준(요약)
- 페르소나 핵심 성향: '{most_important_attr}'
- 제품-페르소나 적합도 및 가격 수용도만 고려해 수치 산출.

# OUTPUT FORMAT
{output_format}
"""
    return system_prompt, user_prompt.strip()


# =============================
# 5) 메인 로직
# =============================
def main():
    # 5-1) 합본 JSONL 로드
    with open(SINGLE_FILE, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f]

    if not records:
        raise RuntimeError("JSONL 레코드가 비어 있습니다.")

    # 5-2) 제품 정보 추출 (첫 레코드 기준 + 유연한 보정)
    product_info = extract_product_info(records[0], SINGLE_FILE)

    print(f"제품 정보 추출 완료: {json.dumps({k: product_info.get(k) for k in PRODUCT_KEYS}, ensure_ascii=False)}")

    # 5-3) 페르소나 리스트 구성
    personas = []
    for rec in records:
        p = extract_persona_info(rec)
        # persona_key가 없으면 순번 부여
        if 'persona_key' not in p:
            p = {**p, 'persona_key': len(personas) + 1}
        personas.append(p)

    print(f"페르소나 {len(personas)}명 불러오기 완료.")

    # 5-4) 경쟁사 맥락 로드
    market_context = load_market_context(COMPETITOR_FILE_CLEANED)

    # 5-5) 프롬프트 생성
    print("프롬프트 생성을 시작합니다...")
    all_advanced_prompts = []
    for persona in personas:
        system_prompt, user_prompt = create_advanced_prompt(product_info, persona, market_context)
        all_advanced_prompts.append({
            "product_name": product_info.get("product_name"),
            "persona_key": persona.get("persona_key"),
            "system_prompt": system_prompt,
            "user_prompt": user_prompt
        })

    # 5-6) 저장
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for item in all_advanced_prompts:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("=" * 40)
    print("🎉 모든 작업이 성공적으로 완료되었습니다!")
    print(f"총 {len(all_advanced_prompts)}개의 프롬프트가 생성되어 '{OUTPUT_FILE}'에 저장되었습니다.")

    # 5-7) 결과 미리보기
    if all_advanced_prompts:
        example = all_advanced_prompts[0]
        print("\n--- 생성된 첫 번째 프롬프트 예시 ---")
        print("\n[SYSTEM PROMPT]")
        print(example['system_prompt'])
        print("\n[USER PROMPT]")
        print(example['user_prompt'])

main()  # 한 번에 실행

제품 정보 추출 완료: {"brand": null, "product_name": "프리미엄 동원참치액 900g_100", "category": null, "features": null, "targeted_consumer": null, "price_text": null, "advertise_info": null}
페르소나 100명 불러오기 완료.
경쟁사 시장 데이터 불러오기 완료. (카테고리: ['RTD_액상커피', '그릭요거트', '참치액', '참치캔', '캔햄'])
프롬프트 생성을 시작합니다...
🎉 모든 작업이 성공적으로 완료되었습니다!
총 100개의 프롬프트가 생성되어 'llm_prompt/프리미엄 동원참치액 900g_prompts_for_llm.jsonl'에 저장되었습니다.

--- 생성된 첫 번째 프롬프트 예시 ---

[SYSTEM PROMPT]
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. 오직 최종 JSON 객체 하나만 출력하세요. 코드블록/설명/추가 텍스트 금지.

[USER PROMPT]
# INSTRUCTION (Compact)
아래 데이터를 바탕으로 **내부적으로만** 분석하세요(사고과정 출력 금지).
최종 출력은 위 # OUTPUT FORMAT과 **정확히 동일한 키 구조의 JSON 한 덩어리**만 반환합니다.
- 'purchase_probability_pct'는 0~100 **숫자**
- 'monthly_avg_purchase_qty'는 0 이상 **숫자**
- reason은 200자 이하 한 줄, 줄바꿈 금지.

# INPUT DATA
## 1) 제품
{
    "brand": null,
    "product_name": "프리미엄 동원참치액 900g_100",
    "category": null,
    "features": null,
    "targeted_consumer": null,
    "price_text": null,
    "advertise_info": null
}

## 2) 페르소나
{
    